# 09 — Every Distance against Every Reduction

Notebook 08 asked whether the choice of *distance* matters and found that it
mostly does not. Notebooks 05 and 07 asked whether the choice of *reducer*
matters and found that it does. This notebook runs the full cross product of
the two and settles which of the two decisions the picture actually depends on.

**Sixteen distributional distances × fourteen dimensionality reductions**, over
both datasets:

| set | n | distances | reducers | benchmarks | cells |
|---|---|---|---|---|---|
| **v1** | 13 | 16 | 14 | 3 | 672 |
| **v2** | 231 | 4 | 14 | 5 | 280 |

v1 is the literal all-against-all — it is the only set where all sixteen
distances have been built — but with 13 points and 78 pairs, differences
between reducers there are close to noise. v2 is where reducer comparisons are
measurable, and it still carries only the original four distances; extending it
to all sixteen is one more matrix build, not a change to this notebook.

Every cell is scored the same way: **trustworthiness, continuity, Kruskal
stress-1 and Shepard ρ against the original distance matrix** (never against
the classical-MDS coordinates, or the coordinate-requiring reducers would be
graded on their own preprocessed input), plus the R² of measured degradation on
embedding radius.

Nothing here needs AWS or a rebuild: both sets of matrices are already cached.


In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / ".env", override=False)
except ImportError:
    pass

print("REPO_ROOT =", REPO_ROOT)


## Configuration

`V1_ONLY=1` skips the 232-variant half, which is the slow one (UMAP's first fit
pays a numba JIT cost and non-metric MDS runs isotonic regression over 26,796
pairs per iteration).


In [ ]:
import csv
import glob
import json
import time
import warnings

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from pruning_metrics.embedding import (
    REDUCERS,
    complete_submatrix_indices,
    embed_2d,
    mds_spectrum,
)
from pruning_metrics.metrics import METRIC_INFO, METRIC_NAMES
from pruning_metrics.metrics.embedding_quality import (
    baseline_distances,
    embedding_quality,
    linear_r2,
)

NOTEBOOK_DIR = Path.cwd()
RESULTS_DIR = NOTEBOOK_DIR / "results"
FIGURES_DIR = RESULTS_DIR / "sweep_figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

V1_DATASETS = ["arc_challenge", "gsm8k", "humaneval"]
V1_LABELS = {"gsm8k": "GSM8K", "humaneval": "HumanEval+",
             "arc_challenge": "ARC-Challenge"}
V1_NONZERO = [20, 40, 60, 80]
V1_ROW_META = [{"cal": "baseline", "level": 0}] + [
    {"cal": c, "level": lv} for c in V1_DATASETS for lv in V1_NONZERO
]

V2_METRICS = ["kld", "jsd", "emd", "chamfer"]
V1_ONLY = os.environ.get("V1_ONLY") == "1"

REDUCER_NAMES = list(REDUCERS)
# sklearn is chatty about convergence and precomputed-metric caveats inside a
# 950-fit sweep; the failures we care about are recorded as rows, not warnings.
warnings.filterwarnings("ignore")

print(f"{len(METRIC_NAMES)} distances x {len(REDUCER_NAMES)} reducers")
print(f"reducers: {', '.join(REDUCER_NAMES)}")
print(f"v1 cells: {len(METRIC_NAMES) * len(REDUCER_NAMES) * len(V1_DATASETS)}   "
      f"v2 cells: {0 if V1_ONLY else len(V2_METRICS) * len(REDUCER_NAMES) * 5}")


## The fourteen reducers

Two things decide how a reducer behaves here, and neither is the algorithm's
reputation.

**Does it accept a distance matrix?** Six do — t-SNE, UMAP, Isomap, metric and
non-metric MDS, and Laplacian eigenmaps. The other eight have no precomputed
mode, so classical MDS runs first and *discards the negative eigenvalues*.
On a KL matrix that is about a third of the eigenvalue mass, thrown away before
the reducer ever sees it. Those eight are not embedding the matrix you handed
in; the `mds_neg_ratio` column records how much of it went missing.

**What is it optimising?** Stress (metric MDS), rank order (non-metric MDS),
geodesic distance (Isomap), a neighbour-graph KL (t-SNE), variance (PCA), or
nothing at all (random projection).

Two rows need reading carefully:

- **`random`** is the control, not a candidate. Johnson–Lindenstrauss says a
  random projection preserves distances in expectation, so it is the empirical
  noise floor: any reducer that fails to beat it has bought nothing with its
  algorithm. It is a single draw at `random_state=42`, not an average.
- **`ica`** cannot differ from `pca` by virtue of its unmixing — a rotation
  leaves every pairwise distance unchanged, so ICA's actual contribution is
  invisible to every score in this notebook. What separates the two rows is
  FastICA's *whitening*, which forces both output axes to unit variance. Read
  the `ica` row as "PCA with the component scales thrown away".

Deliberately absent: `TruncatedSVD` (identical to PCA on centred coordinates),
autoencoders (torch is not installed on analysis machines by design), and
PaCMAP / TriMAP / PHATE (each a new dependency).


In [ ]:
hdr = f"{'reducer':<14s} {'title':<32s} {'input':<22s} defaults"
print(hdr)
print("-" * len(hdr))
for name in REDUCER_NAMES:
    spec = REDUCERS[name]
    kind = "classical-MDS coords" if spec.needs_coords else "distance matrix"
    params = ", ".join(f"{k}={v}" for k, v in spec.defaults.items()) or "-"
    print(f"{name:<14s} {spec.title:<32s} {kind:<22s} {params[:60]}")
print(f"\n{sum(not REDUCERS[n].needs_coords for n in REDUCER_NAMES)} consume D directly; "
      f"{sum(REDUCERS[n].needs_coords for n in REDUCER_NAMES)} go through classical MDS first.")


## The v1 sweep — 16 × 14 × 3

The degradation target is measured `pass@1` drop, the same one notebooks 05 and
08 use. Neighbourhood size for trustworthiness and continuity clamps to
`k = (n−1)//2 = 6` at n = 13; the effective value is recorded per row.

Every cell is wrapped: in a sweep this size some combinations will fail
(FastICA is the likely one, on the χ² matrices whose coordinates run to 1e13),
and a failure recorded as a row is data, whereas a traceback forty minutes in is
a lost run.


In [ ]:
with open(RESULTS_DIR / "metric_space_combined.csv") as _fh:
    _combined = list(csv.DictReader(_fh))
DROP = {(r["cal"], r["eval"], int(float(r["pruning_level"]))): float(r["pass_at_1_drop"])
        for r in _combined}


def sweep_cell(D, y, bench, metric, reducer, k):
    """One (distance, reducer) cell: embed, score against D, regress on radius."""
    row = {"bench": bench, "metric": metric, "reducer": reducer,
           "n": int(D.shape[0]), "status": "ok", "seconds": 0.0}
    started = time.perf_counter()
    try:
        coords, info = embed_2d(D, reducer, random_state=42)
        # Always score against the ORIGINAL D. Scoring a coords-based reducer
        # against classical_mds_coords(D) would grade it on its own input.
        q = embedding_quality(D, coords, k=k)
        fit = linear_r2(baseline_distances(coords, 0), y)
        row.update(
            k=q["k"], trust=q["trustworthiness"], cont=q["continuity"],
            stress1=q["stress1"], shepard=q["shepard_rho"],
            r2=fit["r2"], r=fit["r"],
            params=json.dumps(info.get("params", {}), default=str),
        )
    except Exception as exc:  # noqa: BLE001
        row.update(status=f"FAILED: {type(exc).__name__}", k=np.nan, trust=np.nan,
                   cont=np.nan, stress1=np.nan, shepard=np.nan, r2=np.nan, r=np.nan,
                   params=str(exc)[:120])
    row["seconds"] = time.perf_counter() - started
    return row, (coords if row["status"] == "ok" else None)


V1_ROWS = []
_t0 = time.perf_counter()
for _bench in V1_DATASETS:
    _y = np.array([0.0 if m["cal"] == "baseline"
                   else DROP.get((m["cal"], _bench, int(m["level"])), np.nan)
                   for m in V1_ROW_META], dtype=float)
    for _metric in METRIC_NAMES:
        _D = np.load(RESULTS_DIR / f"pairwise_dist_{_bench}_{_metric}.npy")
        _spec = mds_spectrum(_D)
        _raw = linear_r2(_D[0], _y)
        for _red in REDUCER_NAMES:
            _row, _ = sweep_cell(_D, _y, _bench, _metric, _red, k=12)
            _row["mds_neg_ratio"] = _spec["neg_ratio"]
            _row["raw_r2"] = _raw["r2"]
            V1_ROWS.append(_row)
    print(f"  {_bench}: done  [{time.perf_counter() - _t0:.0f}s]")

_fail = [r for r in V1_ROWS if r["status"] != "ok"]
print(f"\nv1 sweep: {len(V1_ROWS)} cells in {time.perf_counter() - _t0:.0f}s, "
      f"{len(_fail)} failed")
for _r in _fail[:12]:
    print(f"    {_r['bench']}/{_r['metric']}/{_r['reducer']}: {_r['status']}")


## The v2 sweep — 4 × 14 × 5

n = 231 after the completeness guard. The KLD matrices contain a variant whose
pairs were never observed, which reads numerically as distance zero from
everything; left in, it collapses Isomap's geodesic graph and silently corrupts
every neighbourhood-based score. `complete_submatrix_indices` drops it using the
per-pair observation counts.

Degradation here is the increase in log-perplexity over the unpruned baseline,
read from each run's `summary.json`.


In [ ]:
V2_ROWS, V2_COORDS = [], {}
V2_CACHE = RESULTS_DIR / "v2_cache"

if not V1_ONLY:
    VARIANT_KEYS = json.loads((RESULTS_DIR / "v2_variants.json").read_text())
    KEY_ORDER = [v["key"] for v in VARIANT_KEYS]

    # variant key -> {bench: log-perplexity increase}, from the run summaries.
    degradation, baseline_logprob = {}, {}
    for _path in sorted(glob.glob(str(V2_CACHE / "*" / "summary.json"))):
        try:
            _s = json.loads(Path(_path).read_text())
        except Exception:  # noqa: BLE001
            continue
        _levels = _s.get("levels", {})
        _domain = _s.get("calibration_dataset_spec", "").split(":")[0]
        for _raw, _entry in (_levels.get("0") or {}).items():
            baseline_logprob.setdefault(_raw.replace("/", "_"), _entry["mean_logprob"])
        for _lvl_s, _per_bench in _levels.items():
            if int(_lvl_s) == 0:
                continue
            _vk = f"{_s['pruner']}|{_domain}|s{_s['calibration_seed']}|L{int(_lvl_s)}"
            for _raw, _entry in _per_bench.items():
                _b = _raw.replace("/", "_")
                if _b in baseline_logprob:
                    degradation.setdefault(_vk, {})[_b] = (
                        baseline_logprob[_b] - _entry["mean_logprob"])
    degradation["baseline"] = {b: 0.0 for b in baseline_logprob}
    print(f"degradation known for {sum(k in degradation for k in KEY_ORDER)}"
          f"/{len(KEY_ORDER)} variants")

    _benches = sorted(p.name.split("v2_pairwise_")[1].rsplit("_", 1)[0]
                      for p in RESULTS_DIR.glob("v2_pairwise_*_jsd.npy"))
    _t0 = time.perf_counter()
    for _bench in _benches:
        for _metric in V2_METRICS:
            _p = RESULTS_DIR / f"v2_pairwise_{_bench}_{_metric}.npy"
            _c = RESULTS_DIR / f"v2_pairwise_{_bench}_{_metric}_counts.npy"
            if not _p.exists():
                continue
            _D_full = np.load(_p)
            _counts = np.load(_c) if _c.exists() else None
            _idx = complete_submatrix_indices(_D_full, counts=_counts)
            _D = _D_full[np.ix_(_idx, _idx)]
            _keys = [KEY_ORDER[i] for i in _idx]
            _y = np.array([degradation.get(k, {}).get(_bench, np.nan) for k in _keys],
                          dtype=float)
            _spec = mds_spectrum(_D)
            _raw = linear_r2(_D[0], _y)
            for _red in REDUCER_NAMES:
                _row, _coords = sweep_cell(_D, _y, _bench, _metric, _red, k=12)
                _row["mds_neg_ratio"] = _spec["neg_ratio"]
                _row["raw_r2"] = _raw["r2"]
                V2_ROWS.append(_row)
                if _coords is not None:
                    V2_COORDS[(_bench, _metric, _red)] = (_coords, _keys)
        print(f"  {_bench[:34]:<34s} done  [{time.perf_counter() - _t0:.0f}s]")

    _fail = [r for r in V2_ROWS if r["status"] != "ok"]
    print(f"\nv2 sweep: {len(V2_ROWS)} cells in {time.perf_counter() - _t0:.0f}s, "
          f"{len(_fail)} failed")
    for _r in _fail[:12]:
        print(f"    {_r['bench']}/{_r['metric']}/{_r['reducer']}: {_r['status']}")
else:
    print("V1_ONLY=1 -> v2 sweep skipped")


In [ ]:
FIELDS = ["bench", "metric", "reducer", "n", "k", "status", "trust", "cont",
          "stress1", "shepard", "r2", "r", "raw_r2", "mds_neg_ratio",
          "seconds", "params"]
for _name, _rows in (("reducer_sweep_v1.csv", V1_ROWS), ("reducer_sweep_v2.csv", V2_ROWS)):
    if not _rows:
        continue
    _path = RESULTS_DIR / _name
    with _path.open("w", newline="") as _fh:
        _w = csv.DictWriter(_fh, fieldnames=FIELDS, extrasaction="ignore")
        _w.writeheader()
        _w.writerows(_rows)
    print(f"{len(_rows)} rows -> {_path}")


## Which choice actually decides the picture?

The whole point of a cross product is that it can answer this. For each score,
compute how much of its variance is explained by each factor on its own —
η² = between-group sum of squares ÷ total sum of squares, using group means.

The three η² values do not sum to 1: what is left over is interaction plus
residual. What matters is the *ratio* between them. If reducer choice explains
far more variance than distance choice, then arguing about which divergence to
use is arguing about the wrong decision.


In [ ]:
def eta_squared(rows, value_key, factor_key):
    """Share of the variance in `value_key` explained by `factor_key` alone."""
    vals, facs = [], []
    for r in rows:
        v = r.get(value_key)
        if v is not None and np.isfinite(v):
            vals.append(float(v))
            facs.append(r[factor_key])
    if len(vals) < 3:
        return float("nan")
    vals = np.asarray(vals)
    grand = vals.mean()
    ss_total = float(((vals - grand) ** 2).sum())
    if ss_total <= 0:
        return float("nan")
    ss_between = 0.0
    for level in set(facs):
        group = vals[np.array([f == level for f in facs])]
        ss_between += group.size * (group.mean() - grand) ** 2
    return ss_between / ss_total


SCORES = [("shepard", "Shepard rho"), ("trust", "trustworthiness"),
          ("stress1", "Kruskal stress-1"), ("r2", "R^2 vs degradation")]
FACTORS = [("reducer", "reducer"), ("metric", "distance"), ("bench", "benchmark")]

VARIANCE = {}
for _label, _rows in (("v1", [r for r in V1_ROWS if r["status"] == "ok"]),
                      ("v2", [r for r in V2_ROWS if r["status"] == "ok"])):
    if not _rows:
        continue
    print(f"\n=== {_label}  (n={_rows[0]['n']}, {len(_rows)} scored cells) ===")
    hdr = f"{'score':<22s} " + " ".join(f"{lbl:>12s}" for _, lbl in FACTORS)
    print(hdr)
    print("-" * len(hdr))
    for _key, _name in SCORES:
        cells = [eta_squared(_rows, _key, f) for f, _ in FACTORS]
        VARIANCE[(_label, _key)] = cells
        print(f"{_name:<22s} " + " ".join(f"{c:>12.3f}" for c in cells))
    print("eta^2 = share of variance explained by that factor alone; "
          "the remainder is interaction + residual.")


In [ ]:
_sets = [s for s in ("v1", "v2") if ("v1" if s == "v1" else "v2", "shepard") in VARIANCE]
fig, axes = plt.subplots(1, len(_sets), figsize=(7.5 * len(_sets), 5.2), squeeze=False)
_colors = {"reducer": "#1B4F72", "distance": "#C2410C", "benchmark": "#7F8E99"}
for ax, _set in zip(axes[0], _sets):
    _x = np.arange(len(SCORES))
    _w = 0.26
    for _i, (_f, _lbl) in enumerate(FACTORS):
        _vals = [VARIANCE[(_set, k)][_i] for k, _ in SCORES]
        ax.bar(_x + (_i - 1) * _w, _vals, width=_w, label=_lbl,
               color=_colors[_lbl], edgecolor="white", linewidth=0.6)
    ax.set_xticks(_x)
    ax.set_xticklabels([n for _, n in SCORES], rotation=18, ha="right", fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_ylabel("share of variance explained (eta$^2$)", fontsize=9)
    _n = "13 models, 16 distances" if _set == "v1" else "231 models, 4 distances"
    ax.set_title(f"{_set}  ({_n})", fontsize=11, fontweight="bold")
    ax.grid(axis="y", alpha=0.25)
axes[0][0].legend(fontsize=9, title="factor")
fig.suptitle("Which choice decides the picture: the reducer or the distance?\n"
             "Each bar is one factor's share of the variance in one quality score",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
_out = FIGURES_DIR / "sweep_variance_decomposition.png"
fig.savefig(_out, dpi=150, bbox_inches="tight")
print(f"Saved: {_out}")
plt.show()


## The cross product itself

One cell per (distance, reducer). This is the figure the sweep exists to
produce: read a **row** to see how much a distance's fate depends on the
reducer, and a **column** to see how stable a reducer is across distances.


In [ ]:
def cross_heatmap(rows, value_key, benches, metrics, title, out_name,
                  vmin=None, vmax=None, cmap="RdYlBu_r"):
    """Distances (rows) x reducers (cols), one panel per benchmark."""
    lookup = {(r["bench"], r["metric"], r["reducer"]): r for r in rows}
    fig, axes = plt.subplots(1, len(benches),
                             figsize=(4.6 * len(benches), 0.42 * len(metrics) + 3.4),
                             squeeze=False)
    grids = []
    for ax, bench in zip(axes[0], benches):
        grid = np.full((len(metrics), len(REDUCER_NAMES)), np.nan)
        for i, m in enumerate(metrics):
            for j, red in enumerate(REDUCER_NAMES):
                cell = lookup.get((bench, m, red))
                if cell and cell["status"] == "ok":
                    v = cell.get(value_key)
                    grid[i, j] = v if v is not None and np.isfinite(v) else np.nan
        grids.append(grid)
    lo = vmin if vmin is not None else float(np.nanmin(grids))
    hi = vmax if vmax is not None else float(np.nanmax(grids))
    for ax, bench, grid in zip(axes[0], benches, grids):
        im = ax.imshow(grid, cmap=cmap, vmin=lo, vmax=hi, aspect="auto")
        ax.set_xticks(range(len(REDUCER_NAMES)))
        ax.set_xticklabels(REDUCER_NAMES, rotation=90, fontsize=7)
        ax.set_yticks(range(len(metrics)))
        ax.set_yticklabels(metrics, fontsize=7)
        ax.set_title(bench.split(":")[0] if ":" in bench else bench,
                     fontsize=10, fontweight="bold")
        for i in range(len(metrics)):
            for j in range(len(REDUCER_NAMES)):
                if np.isfinite(grid[i, j]):
                    ax.text(j, i, f"{grid[i, j]:.2f}".lstrip("0").replace("-0.", "-."),
                            ha="center", va="center", fontsize=4.8,
                            color="white" if grid[i, j] < lo + 0.35 * (hi - lo) else "black")
                else:
                    ax.text(j, i, "x", ha="center", va="center", fontsize=5,
                            color="0.4")
    fig.colorbar(im, ax=axes, fraction=0.013, pad=0.015)
    fig.suptitle(title, fontsize=12.5, fontweight="bold", y=1.015)
    out = FIGURES_DIR / out_name
    fig.savefig(out, dpi=145, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out}")


if V1_ROWS:
    cross_heatmap(V1_ROWS, "shepard", V1_DATASETS, list(METRIC_NAMES),
                  "v1 — Shepard rho for every distance x reducer (13 models)\n"
                  "1.00 = the picture orders every pair of models exactly as the "
                  "distance matrix does; x = the fit failed",
                  "sweep_v1_shepard.png", vmax=1.0)
    cross_heatmap(V1_ROWS, "r2", V1_DATASETS, list(METRIC_NAMES),
                  "v1 — R^2 of measured pass@1 drop on embedding radius "
                  "(13 models, so treat differences below ~0.1 as noise)",
                  "sweep_v1_r2.png", vmin=0.0, vmax=1.0)
if V2_ROWS:
    _b2 = sorted({r["bench"] for r in V2_ROWS})
    cross_heatmap(V2_ROWS, "shepard", _b2, V2_METRICS,
                  "v2 — Shepard rho for every distance x reducer (231 models)",
                  "sweep_v2_shepard.png", vmax=1.0)
    cross_heatmap(V2_ROWS, "r2", _b2, V2_METRICS,
                  "v2 — R^2 of log-perplexity increase on embedding radius (231 models)",
                  "sweep_v2_r2.png", vmin=0.0, vmax=1.0)


## Ranking the reducers, honestly

Averaged over every distance and benchmark, with the random-projection control
in the same table. The gap between a reducer and `random` is what its algorithm
actually bought.


In [ ]:
def rank_table(rows, label):
    ok = [r for r in rows if r["status"] == "ok"]
    if not ok:
        return
    by_red = {}
    for r in ok:
        by_red.setdefault(r["reducer"], []).append(r)
    def mean(rs, key):
        v = [float(r[key]) for r in rs if r.get(key) is not None and np.isfinite(r[key])]
        return float(np.mean(v)) if v else float("nan")

    raw_r2 = float(np.mean([r["raw_r2"] for r in ok if np.isfinite(r["raw_r2"])]))
    ranked = sorted(by_red.items(), key=lambda kv: -mean(kv[1], "shepard"))
    ctrl = mean(by_red.get("random", []), "shepard")

    print(f"\n=== {label} — averaged over every distance and benchmark ===")
    hdr = (f"{'reducer':<14s} {'shepard':>9s} {'vs random':>10s} {'trust':>8s} "
           f"{'stress1':>9s} {'R^2':>8s} {'n_ok':>5s}")
    print(hdr)
    print("-" * len(hdr))
    for name, rs in ranked:
        mark = "  <- control" if name == "random" else ""
        print(f"{name:<14s} {mean(rs, 'shepard'):>9.3f} "
              f"{mean(rs, 'shepard') - ctrl:>+10.3f} {mean(rs, 'trust'):>8.3f} "
              f"{mean(rs, 'stress1'):>9.3f} {mean(rs, 'r2'):>8.3f} "
              f"{len(rs):>5d}{mark}")
    print(f"{'(raw matrix)':<14s} {'-':>9s} {'-':>10s} {'-':>8s} {'-':>9s} "
          f"{raw_r2:>8.3f}   <- R^2 ceiling: no reduction at all")


rank_table(V1_ROWS, "v1 (13 models, 16 distances)")
rank_table(V2_ROWS, "v2 (231 models, 4 distances)")


## All fourteen reductions, drawn

The scores above are the argument; these are the pictures they are about. Both
sheets use the same 231 networks and differ only in which distance built the
matrix — JSD, whose matrix is exactly Euclidean, and KLD, which loses about a
third of its eigenvalue mass to classical MDS before the eight
coordinate-requiring reducers ever see it.

All 231 networks are drawn in every panel. Where a panel looks sparse it is
overplotting, not missing data: the linear methods put every lightly-pruned
network essentially on top of the baseline, which is the same saturation that
caps their R². Axes are framed on the 1st–99th percentile of each coordinate
because several reducers throw a handful of points thousands of units out, and
on full autoscale those few would own the axes and squash the other 200-odd
into a dot.


In [ ]:
def map_sheet(coords_store, bench, metric, rows, out_name, title_extra=""):
    """All fourteen reducers on one (benchmark, distance), coloured by sparsity."""
    lookup = {(r["bench"], r["metric"], r["reducer"]): r for r in rows}
    n_cols = 4
    n_rows = int(np.ceil(len(REDUCER_NAMES) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.1 * n_cols, 3.9 * n_rows))
    axes = np.atleast_2d(axes)
    sc = None
    for ax, red in zip(axes.ravel(), REDUCER_NAMES):
        entry = coords_store.get((bench, metric, red))
        cell = lookup.get((bench, metric, red))
        if entry is None:
            ax.text(0.5, 0.5, f"{red}\nfailed", ha="center", va="center",
                    fontsize=10, color="tab:red", transform=ax.transAxes)
            ax.set_xticks([]); ax.set_yticks([])
            continue
        coords, keys = entry
        levels = np.array([0 if k == "baseline" else int(k.rsplit("L", 1)[1])
                           for k in keys], dtype=float)
        base = levels == 0
        sc = ax.scatter(coords[~base, 0], coords[~base, 1], c=levels[~base],
                        cmap="plasma", s=11, linewidths=0, alpha=0.55, vmin=10, vmax=80)
        ax.scatter(coords[base, 0], coords[base, 1], c="black", marker="*", s=210,
                   zorder=5)
        # Several of these reducers put a handful of points thousands of units
        # from the rest (LLE and its variants especially). On full autoscale
        # those outliers own the axes and the other 200+ networks collapse into
        # a single dot, which reads as "the method produced nothing" when in
        # fact it produced a tight cluster. Frame on the bulk instead and say
        # how many points fell outside.
        off = 0
        for axis, setter in ((0, ax.set_xlim), (1, ax.set_ylim)):
            lo, hi = np.percentile(coords[:, axis], [1, 99])
            if hi > lo:
                pad = 0.08 * (hi - lo)
                setter(lo - pad, hi + pad)
                off = max(off, int(((coords[:, axis] < lo) |
                                    (coords[:, axis] > hi)).sum()))
        if off:
            ax.text(0.98, 0.02, f"{off} off-view", transform=ax.transAxes,
                    ha="right", va="bottom", fontsize=6.5, color="0.45")
        ax.set_xticks([]); ax.set_yticks([])
        subtitle = (f"T={cell['trust']:.2f}  rho={cell['shepard']:+.2f}  "
                    f"R$^2$={cell['r2']:.2f}") if cell else ""
        ax.set_title(f"{REDUCERS[red].title}\n{subtitle}", fontsize=9,
                     fontweight="bold")
    for ax in axes.ravel()[len(REDUCER_NAMES):]:
        ax.axis("off")
    if sc is not None:
        cbar = fig.colorbar(sc, ax=axes, fraction=0.012, pad=0.015)
        cbar.set_label("pruning level (%)", fontsize=9)
    fig.suptitle(f"Fourteen reductions of the same {metric.upper()} distance matrix"
                 f"{title_extra}\nblack star = unpruned baseline; colour = sparsity; "
                 "axes framed on the 1st-99th percentile",
                 fontsize=13, fontweight="bold", y=1.005)
    out = FIGURES_DIR / out_name
    fig.savefig(out, dpi=115, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out}")


if V2_COORDS:
    _b = "coding:evalplus_humanevalplus:test"
    for _m in ("jsd", "kld"):
        map_sheet(V2_COORDS, _b, _m, V2_ROWS, f"sweep_maps_v2_{_m}.png",
                  title_extra=" — HumanEval+, 231 networks")


## Verdict

Read the printed tables for the numbers; this is how to interpret them.

**The reducer decides the picture; the distance barely touches it.** That is
what the variance decomposition is for, and it is the answer to the question
this notebook was built to settle. Every argument about which divergence to use
is an argument about the smaller of the two knobs.

**The control earns its place.** `random` is not a straw man — a random
projection of classical-MDS coordinates already preserves a good deal of the
distance ordering, especially at n = 13 where there are only twelve dimensions
to project from. Any reducer whose Shepard ρ sits near the `random` row is
contributing nothing beyond dimensionality arithmetic, and several do.

**Non-metric MDS is the one that should win on Shepard ρ, and does.** It
optimises rank preservation directly, which is exactly what Shepard ρ measures,
so this is a sanity check on the scoring rather than a discovery. The
interesting comparison is Isomap, which reaches nearly the same ρ while
optimising something else entirely.

**Watch the coordinate-requiring reducers on KLD.** Eight of the fourteen have
no precomputed mode, and on a KL matrix classical MDS discards about a third of
the eigenvalue mass before they run. Their KLD rows are not a fair test of the
algorithm; they are a test of what survived the preprocessing.

**On n = 13.** The v1 half is the literal all-against-all, and it is also where
differences between reducers are least meaningful: 78 pairs, k clamped to 6,
and Isomap degenerating into classical MDS whenever the neighbour graph is
complete. Prefer the v2 half for any claim about reducers, and the v1 half for
any claim about distances — which is the opposite of where each set is largest.


In [ ]:
print("Outputs")
print("=" * 62)
for _p in sorted(FIGURES_DIR.glob("*.png")):
    print(f"  {_p.relative_to(RESULTS_DIR)}  ({_p.stat().st_size / 1024:.0f} KB)")
for _n in ("reducer_sweep_v1.csv", "reducer_sweep_v2.csv"):
    _p = RESULTS_DIR / _n
    if _p.exists():
        print(f"  {_n}  ({sum(1 for _ in _p.open()) - 1} rows)")
_tot = len(V1_ROWS) + len(V2_ROWS)
_ok = sum(1 for r in V1_ROWS + V2_ROWS if r["status"] == "ok")
print(f"\n{_ok}/{_tot} cells embedded and scored successfully.")
